# Developing with OpenAI: AIM Edition

## Exploring LLM Prompting Strategies for Economic Reasoning  
### *Inflation & Interest Rate Case Study*

This notebook investigates how different prompting strategies (zero-shot, few-shot, reasoning vs non-reasoning models) affect the ability of large language models (LLMs) to reason about inflation, interest rates, and overall market dynamics.  

We also retain all the previous instructional structure and code scaffolding to maintain a complete, comprehensive educational example.

## 1. Getting Started

The first thing we'll do is load the [OpenAI Python Library](https://github.com/openai/openai-python/tree/main)!

In [ ]:
# Used for Google Colab
#!pip install openai -q


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip



## Discussion and Problem Framing

We aim to answer:  
> *"What is the best prompting approach and model type to understand how the market is performing today?"*  

### Types of LLM Tasks Involved

| Type | Description | Example Output |
|------|--------------|----------------|
| **Retrieval** | Factual recall | “Inflation in 2025 is around 3.1% in the U.S.” |
| **Reasoning** | Logical chain between variables | “Higher inflation led the Fed to raise rates → borrowing costs rose → slower GDP.” |
| **Generation** | Narrative creation / summary | “The market shows cooling signals despite moderate inflation…” |

Each prompt and model will be evaluated on reasoning depth, factual correctness, and structure quality.


### Used models in this repo

| Rank | Model Name | Primary Purpose | OpenAI's Official Claim |
|------|------------|-----------------|------------------------|
| 1 | **GPT-5** | Advanced reasoning for complex economic analysis | Uses a dynamic router that chooses between quick responses and deeper 'thinking' when needed; performs at PhD-level across domains |
| 2 | **GPT-4.1** | Enhanced coding and long-context comprehension | Offers significant advancements in coding capabilities, long context comprehension (up to 1M tokens), and instruction following |
| 3 | **GPT-4-turbo** | General-purpose non-reasoning model for structured responses | Improved version of GPT-4 with enhanced performance, lower latency, and updated knowledge cutoff |
| 4 | **GPT-4o-mini** | Fast, efficient model for quick responses | Cost-efficient AI model designed to make advanced AI technology more affordable and accessible |


## 2. Setting Environment Variables

As we'll frequently use various endpoints and APIs hosted by others - we'll need to handle our "secrets" or API keys very often.

We'll use the following pattern throughout this bootcamp - but you can use whichever method you're most familiar with.

In [ ]:
# For Google Colab
# import os
# import getpass

# os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key")

In [1]:
# For local development
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


## 3. Using the OpenAI Python Library

Let's jump right into it!

> NOTE: You can, and should, reference OpenAI's [documentation](https://platform.openai.com/docs/api-reference/authentication?lang=python) whenever you get stuck, have questions, or want to dive deeper.

### Creating a Client

The core feature of the OpenAI Python Library is the `OpenAI()` client. It's how we're going to interact with OpenAI's models, and under the hood of a lot what we'll touch on throughout this course.

> NOTE: We could manually provide our API key here, but we're going to instead rely on the fact that we put our API key into the `OPENAI_API_KEY` environment variable!

In [2]:
from openai import OpenAI

client = OpenAI()

### Using the Client

Now that we have our client - we're going to use the `.chat.completions.create` method to interact with the model.

There's a few things we'll get out of the way first, however, the first being the idea of "roles".

First it's important to understand the object that we're going to use to interact with the endpoint. It expects us to send an array of objects of the following format:

```python
{"role" : "ROLE", "content" : "YOUR CONTENT HERE", "name" : "THIS IS OPTIONAL"}
```

Second, there are three "roles" available to use to populate the `"role"` key:

- `system`
- `assistant`
- `user`

OpenAI provides some context for these roles [here](https://help.openai.com/en/articles/7042661-moving-from-completions-to-chat-completions-in-the-openai-api).

We'll explore these roles in more depth as they come up - but for now we're going to just stick with the basic role `user`. The `user` role is, as it would seem, the user!

Thirdly, it expects us to specify a model!

We'll use the `gpt-5-mini` model as stated above.

Let's look at an example!



In [4]:
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "Hello!"}]
)

Let's look at the response object.

In [5]:
response

ChatCompletion(id='chatcmpl-CaMv4wFYLfz2yVUClJIOqYRpxNDpP', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hi there! How can I help you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1762783946, model='gpt-5-mini-2025-08-07', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=19, prompt_tokens=8, total_tokens=27, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [6]:
print(response.choices[0].message.content)

Hi there! How can I help you today?


>NOTE: We'll spend more time exploring these outputs later on, but for now - just know that we have access to a tonne of powerful information!

### System Role

Now we can extend our prompts to include a system prompt.

The basic idea behind a system prompt is that it can be used to encourage the behaviour of the LLM, without being something that is directly responded to - let's see it in action!

In the newest OpenAI API, the **system message** still defines the model’s behavior.  
Sometimes it is referred to as an *instruction block*.

Example system prompt for our economics case:

In [7]:
system_prompt = """
You are an experienced economic analyst explaining how inflation and interest rates interact.   
Use 2025 U.S. market context when relevant.
Your answer should not exceed 5 sentences. 
"""
print(system_prompt)

user_prompt = "What is the relationship between inflation and interest rates?"
print(user_prompt)

list_of_prompts = [

    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

irate_response = client.chat.completions.create(
    model="gpt-4-turbo",
    messages=list_of_prompts
)

print(irate_response.choices[0].message.content)


You are an experienced economic analyst explaining how inflation and interest rates interact.   
Use 2025 U.S. market context when relevant.
Your answer should not exceed 5 sentences. 

What is the relationship between inflation and interest rates?
Inflation and interest rates are closely linked in economic policy, primarily through central bank actions such as those of the Federal Reserve. When inflation is high, central banks may increase interest rates to cool off the economy by making borrowing more expensive, thereby reducing spending and slowing inflation. Conversely, during periods of low inflation or deflation, interest rates may be lowered to encourage borrowing and spending, which can help stimulate the economy. As of 2025, if the U.S. is experiencing inflationary pressures, the Federal Reserve might raise interest rates to try to stabilize prices, balancing growth with the goal of keeping inflation at its target level. This interaction emphasizes the role of monetary policy

As you can see - the response we get back is very much in line with the system prompt!

Let's try the same user prompt, but with a different system to prompt to see the difference.

In [8]:
system_prompt = """
You are a cool and fun elementary teacher explaining to 6-year olds how inflation and interest rates interact.   
Use 2025 U.S. market context when relevant.
Your answer should not exceed 5 sentences.
"""
print(system_prompt)

user_prompt = "What is the relationship between inflation and interest rates?"
print(user_prompt)

list_of_prompts = [

    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

irate_response = client.chat.completions.create(
    model="gpt-4-turbo",
    messages=list_of_prompts
)

print(irate_response.choices[0].message.content)


You are a cool and fun elementary teacher explaining to 6-year olds how inflation and interest rates interact.   
Use 2025 U.S. market context when relevant.
Your answer should not exceed 5 sentences.

What is the relationship between inflation and interest rates?
Alright, imagine you have a piggy bank where you save your allowance to buy toys. Now, if the prices of toys start going up (that's what we call inflation), your money buys less. To help people save more money when things are getting expensive, banks can make a decision to increase the amount of extra money they give you for keeping your money with them - this extra money is what we call interest. So, when inflation goes up, usually the interest rates go up too, because it encourages people to save rather than spend, and that can help slow down how fast prices are rising. In 2025, if toys and other things start costing more, expect the banks to possibly give more interest if you save your money with them!


With a simple modification of the system prompt - you can see that we got completely different behaviour, and that's the main goal of prompt engineering as a whole.

Also, congrats, you just engineered your first prompt!

### Few-shot Prompting

Now that we have a basic handle on the `system` role and the `user` role - let's examine what we might use the `assistant` role for.

The most common usage pattern is to "pretend" that we're answering our own questions. This helps us further guide the model toward our desired behaviour. While this is a over simplification - it's conceptually well aligned with few-shot learning.

In [9]:
# Zero-shot prompt
prompt_zero = "Explain how inflation affects interest rate decisions."
list_of_prompts = [
    {"role": "user", "content": prompt_zero}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=list_of_prompts
)

print('zero-shot response:', response.choices[0].message.content)

zero-shot response: Inflation significantly influences interest rate decisions made by central banks and other financial institutions. Here's an overview of how that relationship works:

### 1. **Understanding Inflation**
Inflation refers to the general increase in prices of goods and services in an economy over a period. It decreases the purchasing power of money, meaning consumers can buy less with the same amount of money as prices rise.

### 2. **Central Bank Mandate**
Central banks, like the Federal Reserve in the United States, often have a dual mandate: to promote maximum employment and to ensure price stability, which often translates to controlling inflation. A common target for inflation is around 2%, as moderate inflation is seen as a sign of a growing economy.

### 3. **Raising Interest Rates**
- **Counteracting High Inflation**: When inflation is rising above the target level, central banks may decide to raise interest rates. Higher interest rates typically make borrowing 

In [10]:
# Few-shot prompt template

question = "Explain how inflation affects interest rate decisions."

few_shot_prompt = f"""
Example 1:
Q: The price of pizza slices jumps from $2 to $4. What might the central bank do?
A: They turn down the oven heat 🍕🔥 — raise interest rates so people buy fewer slices and cool off the price party.

Example 2:
Q: Interest rates drop and borrowing gets cheaper. What happens at Snack City?
A: Everyone's grabbing extra fries and milkshakes 🍟🥤— cheap credit means more spending, which can make prices rise again.

Now answer:
Q: {question}
"""

list_of_prompts = [
    {"role": "user", "content": few_shot_prompt}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=list_of_prompts
)

print('few-shot response:', response.choices[0].message.content)

few-shot response: A: When inflation rises like hot air balloons 🎈, central banks often pull the strings on interest rates to bring them back down to earth. Higher inflation means people are paying more for goods and services, so the central bank might decide to raise interest rates. This makes borrowing more expensive and encourages savings, cooling off spending and eventually putting a lid on those rising prices. If inflation is low and stable, the central bank might lower rates to encourage borrowing and spending, helping keep the economy buzzing along smoothly.


### Helper functions

We're going to create some helper functions to aid in using the OpenAI API - just to make our lives a bit easier.

> NOTE: Take some time to understand these functions between class!

In [11]:
from IPython.display import display, Markdown

def get_response(client: OpenAI, messages: list, model: str = "gpt-4o-mini") -> str:
    return client.chat.completions.create(
        model=model,
        messages=messages
    )

def system_prompt(message: str) -> dict:
    return {"role": "system", "content": message}

def assistant_prompt(message: str) -> dict:
    return {"role": "assistant", "content": message}

def user_prompt(message: str) -> dict:
    return {"role": "user", "content": message}

def pretty_print(message: str) -> str:
    display(Markdown(message.choices[0].message.content))

Different way we can do prompting -> using the helper's functions

In [12]:
# Now, show the economic example with both user and assistant prompts
few_shot_prompts = [
    user_prompt("Inflation rises fast. How does the central bank react — dating analogy please!"),
    assistant_prompt("They play hard to get — raise rates — to cool off the economy's over-eager spending habits."),

    user_prompt("What happens when interest rates are too low for too long?"),
    assistant_prompt("Everyone gets too comfortable — too many relationships (loans) form, and eventually hearts (bubbles) break."),

    user_prompt("Explain deflation using a dating metaphor."),
    assistant_prompt("No one's asking anyone out — everyone waits for a better deal, so the economy gets lonely and quiet."),
    # 👇 Here's the actual question we want the model to answer
    user_prompt("Describe quantitative easing")
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=few_shot_prompts
)

print(response.choices[0].message.content)


Quantitative easing is like a generous friend who starts giving out gifts (money) to encourage everyone to mingle and have fun again. They hope that by boosting the party atmosphere, more people will date, spend, and invest, sparking excitement and activity in the economy.


### 🏗️ Activity #1:
Mission:
Experiment with how different prompt structures, system, user, and assistant, plus zero-shot and few-shot prompting, can transform an AI’s response.
Your goal: craft the most effective prompt and see how GPT-4-Turbo reacts!

You’ll test how GPT-4-Turbo behaves under four different setups:
1. System/User roles only (Zero-shot)
2. System/User roles + examples (Few-shot)
3. No system role at all (User only)
4. Creative system prompt twist



### Chain of Thought Prompting

We'll head one level deeper and explore the world of Chain of Thought prompting (CoT).

This is a process by which we can encourage the LLM to handle slightly more complex tasks.

Let's look at a simple reasoning based example without CoT.

In [13]:
reasoning_problem = """
The central bank increases the policy rate by 1.5 pp in response to 5 % inflation while nominal wage growth is 3 %.
What happens to real wages?
"""

list_of_prompts = [
    user_prompt(reasoning_problem)
]

reasoning_response = get_response(client, list_of_prompts)
pretty_print(reasoning_response)

To understand what happens to real wages when the central bank increases the policy rate, we first need to define a few terms:

1. **Nominal Wage Growth**: This is the rate at which wages are increasing in nominal terms (without adjusting for inflation). In this case, nominal wage growth is 3%.

2. **Inflation Rate**: This is the rate at which the general price level of goods and services is rising. In this case, inflation is at 5%.

3. **Real Wages**: Real wages account for inflation and represent the purchasing power of wages. Real wages can be calculated using the formula:

   \[
   \text{Real Wages} = \frac{\text{Nominal Wages}}{(1 + \text{Inflation Rate})}
   \]

Given the nominal wage growth of 3% (0.03) and an inflation rate of 5% (0.05), we can calculate the change in real wages:

1. **Nominal Wage Growth**: Let's say the initial wage is 100. After a nominal wage growth of 3%, the new wage becomes:
   \[
   \text{New Wage} = 100 \times (1 + 0.03) = 103
   \]

2. **Calculating Real Wage Growth**: The real wage growth can be found by adjusting the nominal wage for inflation:
   \[
   \text{Real Wage Growth} \approx \text{Nominal Wage Growth} - \text{Inflation Rate}
   \]
   \[
   \text{Real Wage Growth} \approx 0.03 - 0.05 = -0.02
   \]

So, the real wages would decrease by approximately 2%.

In summary, when the central bank increases the policy rate by 1.5 percentage points in response to 5% inflation while nominal wage growth is at 3%, the real wages decrease, resulting in a reduction in purchasing power.

Let's see if we can leverage a simple CoT prompt to improve our model's performance on this task:

In [14]:
list_of_prompts = [
    user_prompt(reasoning_problem + "Think step-by-step about how nominal wages, prices, and interest rates interact through the labor market and aggregate demand. Then explain the real wage effect.")
]

reasoning_response = get_response(client, list_of_prompts)
pretty_print(reasoning_response)

To understand the interaction between nominal wages, prices, and interest rates, as well as their impact on real wages, let's break this down step by step.

### Definitions and Concepts

1. **Nominal Wage**: The wage paid to workers that does not account for inflation; in this case, it has grown by 3%.

2. **Price Level/Inflation**: The rate at which the general level of prices for goods and services is rising; here, it is at 5%.

3. **Real Wage**: This reflects the purchasing power of nominal wages, calculated by adjusting nominal wages for inflation. The formula for real wage is:
   \[
   \text{Real Wage} = \frac{\text{Nominal Wage}}{(1 + \text{Inflation Rate})}
   \]

4. **Policy Rate**: The interest rate set by the central bank, which influences other interest rates in the economy. An increase in the policy rate typically aims to reduce inflation.

### Step-by-step Analysis

1. **Initial Conditions**: We have 3% nominal wage growth and 5% inflation. This means that while wages are increasing, prices are increasing even faster.

2. **Calculate Real Wage**: Given the nominal wage growth of 3% (0.03), the inflation rate of 5% (0.05), we can calculate the real wage:
   \[
   \text{Real Wage} = \frac{1 + 0.03}{1 + 0.05} = \frac{1.03}{1.05} \approx 0.98095
   \]
   This indicates that real wages are effectively falling, as they are less than 1. 

3. **Understanding the Impact of the Policy Rate Increase**: When the central bank increases the policy rate by 1.5 percentage points (pp), it aims to reduce aggregate demand by making borrowing more expensive. This can contribute to slower economic growth and potentially lower inflation in the future, as businesses and consumers cut back on spending.

4. **Labor Market Response**: As aggregate demand weakens due to higher interest rates:
   - Employment growth may slow, or even lead to layoffs.
   - The demand for labor could decrease, leading to downward pressure on wage growth or even wage reductions in the future.

5. **Inflation Dynamics**: The initial impact from the rate hike may take time to unfold in the economy. If inflation expectations are anchored, the effect may stabilize prices in the medium term, potentially slowing down price increases.

### Conclusion on Real Wages

Based on the above:

- **Real wages are falling**: With nominal wages increasing by 3% but inflation at 5%, the purchasing power of wages decreases.
- The increase in the policy rate is a response to inflation and designed to stabilize prices in the long term, but in the short-term, real wages have decreased as they are not keeping pace with inflation.

Thus, while the increase in the policy rate aims to combat inflation, it does not immediately improve real wages, which are already reduced due to higher inflation.


## 3. Running Comparative Experiment

We'll test combinations of model type (reasoning vs non-reasoning) and prompting style (zero-shot vs few-shot).


In [15]:
# --------------------------------------------------
# 🧩 Comparing GPT Models: Reasoning vs Non-Reasoning
# --------------------------------------------------

from openai import OpenAI
client = OpenAI()

system_prompt = """
You are an experienced economic analyst.
"""

question = """What is the impact of inflation on real wages? Respond in a concise manner."""

prompt_few = f"""
Use this exact format to answer the question:
Example 1:
{{
  "possible_explanation": "Wage catch-up effect",
  "mechanism": "Workers negotiate higher nominal wages to preserve purchasing power as prices rise.",
  "impact_on_wages": "Nominal wages increase roughly in line with inflation, keeping real wages stable in the short run.",
  "time_frame": "Short to medium run",
  "economic_context": "Inflationary periods with strong labor bargaining power or cost-of-living adjustments."
}}

Example 2:
{{
  "possible_explanation": "Real wage erosion",
  "mechanism": "When nominal wages lag behind price growth, workers lose purchasing power.",
  "impact_on_wages": "Real wages decline despite nominal wage increases, reducing workers’ living standards.",
  "time_frame": "Immediate term",
  "economic_context": "High inflation environments with weak wage indexation or rigid labor contracts."
}}

Now answer:
Q: {question}
"""


# --------------------------------------------------
# MODEL 1: GPT-4-turbo  → Non-Reasoning
# --------------------------------------------------
print("\n==============================")
print("MODEL 1: GPT-4-turbo (Non-Reasoning)")
print("==============================\n")

# Zero-shot
answer_nonreasoning_zero_shot = client.chat.completions.create(
    model="gpt-4-turbo",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ],
)
print("Zero-Shot Prompting (no examples):\n")
print("A:", answer_nonreasoning_zero_shot.choices[0].message.content, "\n")



MODEL 1: GPT-4-turbo (Non-Reasoning)

Zero-Shot Prompting (no examples):

A: Inflation erodes the purchasing power of money, which means that if nominal wage increases do not keep up with the rate of inflation, real wages (which reflect the purchasing power of wages) decrease. This leads to workers being able to afford fewer goods and services with their income, effectively reducing their standard of living if wage hikes lag behind inflation. 



In [16]:
# --------------------------------------------------
# MODEL 2: GPT-5  → Reasoning
# --------------------------------------------------
print("\n==============================")
print("MODEL 2: GPT-5 (Reasoning-Tuned)")
print("==============================\n")

# Zero-shot
answer_reasoning_zero_shot = client.chat.completions.create(
    model="gpt-5",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ],
)
print("Zero-Shot Prompting (no examples):\n")
print("A:", answer_reasoning_zero_shot.choices[0].message.content, "\n")



MODEL 2: GPT-5 (Reasoning-Tuned)

Zero-Shot Prompting (no examples):

A: - Real wage ≈ nominal wage growth minus inflation. If prices rise faster than pay, purchasing power falls.
- When inflation equals wage growth, real wages are flat; if wages outpace inflation, real wages rise.
- Unexpected inflation typically reduces real wages (wages adjust slowly), shifting income from workers to employers/debtors in the short run.
- Indexation and frequent bargaining can offset inflation; without them, lower-paid workers often see larger real wage erosion.
- Long run: real wages track productivity, not inflation per se, but high/volatile inflation disrupts bargaining and planning. 



In [18]:
print("\n==============================")
print("MODEL 1: GPT-4-turbo (Non-Reasoning)")
print("==============================\n")

# Few-shot
answer_nonreasoning_few_shot = client.chat.completions.create(
    model="gpt-4-turbo",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt_few}
    ],
)
print("Few-Shot Prompting (with examples):\n")
print("A:", answer_nonreasoning_few_shot.choices[0].message.content, "\n")


MODEL 1: GPT-4-turbo (Non-Reasoning)

Few-Shot Prompting (with examples):

A: {
  "possible_explanation": "Real wage decline",
  "mechanism": "Inflation causes general price levels to increase, but if wage growth does not keep pace, purchasing power decreases.",
  "impact_on_wages": "Nominal wages may rise, but not sufficiently to match inflation, leading to a decrease in real wages.",
  "time_frame": "Immediate to short term",
  "economic_context": "Periods of rapid or unexpected inflation where wage adjustments are not timely or adequate."
} 



In [19]:
print("\n==============================")
print("MODEL 2: GPT-5 (Reasoning-Tuned)")
print("==============================\n")

# Few-shot
answer_reasoning_few_shot = client.chat.completions.create(
    model="gpt-5",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt_few}
    ],
)
print("Few-Shot Prompting (with examples):\n")
print("A:", answer_reasoning_few_shot.choices[0].message.content, "\n")


MODEL 2: GPT-5 (Reasoning-Tuned)

Few-Shot Prompting (with examples):

A: Example 1:
{
  "possible_explanation": "Wage catch-up (indexation)",
  "mechanism": "Workers secure COLAs or negotiate higher nominal pay to offset price increases.",
  "impact_on_wages": "Nominal wages rise near inflation, keeping real wages broadly stable after a lag.",
  "time_frame": "Short to medium run",
  "economic_context": "Inflation with strong bargaining power or explicit indexation."
}

Example 2:
{
  "possible_explanation": "Real wage erosion",
  "mechanism": "Inflation outpaces nominal wage growth, cutting purchasing power.",
  "impact_on_wages": "Real wages fall despite nominal increases.",
  "time_frame": "Immediate term",
  "economic_context": "High inflation with weak indexation or rigid contracts."
} 




## 4. Evaluation Framework

LLM as a judge


In [20]:
import json

# --------------------------------------------------
# ⚖️ LLM-as-a-Judge Evaluation Script
# --------------------------------------------------

# Define evaluation scale (0–4)
# 0 = completely incorrect / irrelevant
# 1 = partially correct but weak or inaccurate reasoning
# 2 = fair factual accuracy, minimal reasoning
# 3 = accurate and somewhat reasoned
# 4 = highly accurate, clear causal explanation, correct logic

evaluation_prompt = f"""
You are an impartial economics teacher grading two student answers to the same question.

Question:
{question}

Answer A (non-reasoning model):
{answer_nonreasoning_few_shot.choices[0].message.content}

Answer B (reasoning model):
{answer_reasoning_few_shot.choices[0].message.content}

Evaluate both answers on accuracy and reasoning quality on a 0–4 scale:
- 0 = completely incorrect or irrelevant
- 1 = partially correct, but flawed
- 2 = fair factual accuracy, limited reasoning
- 3 = mostly correct, some reasoning
- 4 = fully accurate and clearly reasoned, ability to see the interdependencies between variables.

Return your evaluation as a JSON object in this exact format:
{{
  "Answer A Score": <0-4>,
  "Answer B Score": <0-4>,
  "Better Answer": "A" or "B",
  "Explanation": "Why the better answer is more accurate or reasoned"
}}
"""

# Choose a strong evaluator model (GPT-4.1 is good for judging)
evaluation = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are an impartial LLM evaluator for economics-related answers."},
        {"role": "user", "content": evaluation_prompt}
    ],
)

# Parse and display the evaluation
response_text = evaluation.choices[0].message.content

# Optional: try to parse JSON for structured output
try:
    result = json.loads(response_text)
    print("\nParsed JSON Result:")
    print(json.dumps(result, indent=2))
except json.JSONDecodeError:
    print("\nNote: Could not parse JSON, model may have returned free text instead.")



Parsed JSON Result:
{
  "Answer A Score": 3,
  "Answer B Score": 4,
  "Better Answer": "B",
  "Explanation": "Answer A is correct in stating the common outcome (real wages fall if nominal wages don't keep up) but is simplistic and omits alternative outcomes and relevant mechanisms (indexation, bargaining, expected vs. unexpected inflation). Answer B is more complete and better reasoned: it presents both plausible outcomes (wage catch-up via indexation or bargaining, and real wage erosion when inflation outpaces nominal wages), notes timing differences and economic contexts, and therefore captures the interdependencies between inflation, nominal wage setting, and real wages."
}


### 🏗️ Activity #2:

Evaluate different prompting strategies using your own example.

## Saving results

In [21]:
# Create markdown content
markdown_content = f"""
# 🧠 Reasoning Model Answer
### Question:
How does inflation affect interest rates and the broader market?

### Model Used:
`gpt-4.1` (Reasoning-tuned)

### Response:
{answer_reasoning_few_shot.choices[0].message.content}

---

*This answer was generated by a reasoning model to illustrate step-by-step economic reasoning.*
"""

output_path='./results.md'
# Save to file
with open(output_path, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"✅ Reasoning model answer saved to: {os.path.abspath(output_path)}")

✅ Reasoning model answer saved to: /Users/stringfellowmk/Projects/AIMakerspace/AIE01-MKS/Session_01_LLM_APIs_&_AI-Assisted_Development/results.md


## Conclusion

- **Few-shot prompts** improve structure and reasoning consistency.  
- **Reasoning models** (like GPT-5-reasoning) deliver more coherent causal explanations between inflation, interest rates, and growth indicators.  
- **Non-reasoning models** (e.g., GPT-5-mini) provide faster, surface-level insights ideal for retrieval or summarization tasks.  
- Future work could add **RAG pipelines** with real-time macroeconomic data or integrate with financial dashboards for live LLM reasoning visualization.